# Model Monitoring & CloudWatch Dashboard
This notebook operationalizes the project monitoring requirements. We will implement **Data Quality**, **Model Quality**, and **Infrastructure** monitors, and consolidate them into a unified **CloudWatch Dashboard**.

In [ ]:
!pip install -U "sagemaker<3.0.0"


In [ ]:
import os
import boto3
import sagemaker
import pandas as pd
import time
from datetime import datetime, timedelta
from sagemaker.model_monitor import (
    DefaultModelMonitor,
    ModelQualityMonitor,
    DataCaptureConfig,
    CronExpressionGenerator,
    EndpointInput,
    DatasetFormat
)
from sagemaker.model import Model
from sagemaker.predictor import Predictor
from sagemaker.serializers import CSVSerializer

sess = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = sess.default_bucket()
region = sess.boto_region_name
prefix = "nyc-collisions-monitoring"

print(f"Monitoring initialized in {region} for bucket {bucket}")

## 1. Deploy Real-Time Endpoint
To use SageMaker Model Monitor, we need a live endpoint with **Data Capture** enabled. This captures all inference requests and saves them to S3 for analysis.

In [ ]:
# 1. Define S3 Capture Path
data_capture_uri = f"s3://{bucket}/{prefix}/datacapture"

# 2. Dynamically fetch the latest model from the project prefix
sm_client = boto3.client("sagemaker")
training_jobs = sm_client.list_training_jobs(
    NameContains="sagemaker-scikit-learn",
    StatusEquals="Completed",
    SortBy="CreationTime",
    SortOrder="Descending"
)

if training_jobs["TrainingJobSummaries"]:
    latest_job_name = training_jobs["TrainingJobSummaries"][0]["TrainingJobName"]
    model_artifact = sm_client.describe_training_job(TrainingJobName=latest_job_name)["ModelArtifacts"]["S3ModelArtifacts"]
    print(f"Found latest model: {model_artifact}")
else:
    raise ValueError("No completed training jobs found to deploy.")

# 3. Create Model Entity
image_uri = sagemaker.image_uris.retrieve(framework="sklearn", version="1.2-1", region=region)
model = Model(
    image_uri=image_uri, 
    model_data=model_artifact, 
    role=role,
    entry_point='sagemaker_train.py',
    source_dir='../src'
)

# 4. Setup Data Capture
capture_config = DataCaptureConfig(
    enable_capture=True, 
    sampling_percentage=100, 
    destination_s3_uri=data_capture_uri
)

# 5. Deploy
endpoint_name = f"collisions-monitor-ep-{int(time.time())}"
model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=endpoint_name,
    data_capture_config=capture_config
)

predictor = Predictor(endpoint_name=endpoint_name, serializer=CSVSerializer())
print(f"Endpoint {endpoint_name} is LIVE with Data Capture.")

## 2. Data Quality Monitoring
We will baseline the validation data to identify **Data Drift** (e.g., if Borough distributions change).

In [ ]:
data_monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600,
)

# Use the validation split from Notebook 04
training_prefix = "aai-540-group6/nyc-collisions-ml"
baseline_data_uri = f"s3://{bucket}/{training_prefix}/validation/val.csv"
print(f"Using baseline data: {baseline_data_uri}")

data_monitor.suggest_baseline(
    baseline_dataset=baseline_data_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=f"s3://{bucket}/{prefix}/data_quality_results",
    wait=True
)

## 3. Model Quality Monitoring
We will track **Precision, Recall, and Accuracy** by comparing endpoint predictions against ground truth.

In [ ]:
model_quality_monitor = ModelQualityMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.large",
    sagemaker_session=sess
)

# 1. Create a synthetic baseline with ground truth labels
baseline_results_uri = f"s3://{bucket}/{prefix}/model_quality_results"

model_quality_monitor.suggest_baseline(
    baseline_dataset=baseline_data_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=baseline_results_uri,
    problem_type="BinaryClassification",
    ground_truth_attribute="target", # Our CSV has a 'target' column
    inference_attribute="prediction",
    wait=True
)

## 4. CloudWatch Dashboard Implementation
We will now programmatically create a CloudWatch dashboard that tracks:
1. **Infrastructure:** Endpoint CPU & Memory.
2. **Data Quality:** Baseline violations.
3. **Model Quality:** Accuracy & Recall metrics.

In [ ]:
cw_client = boto3.client('cloudwatch')

dashboard_body = {
    "widgets": [
        {
            "type": "metric",
            "x": 0, "y": 0, "width": 12, "height": 6,
            "properties": {
                "metrics": [
                    ["AWS/SageMaker", "CPUUtilization", "EndpointName", endpoint_name, "VariantName", "AllInstances"]
                ],
                "period": 300,
                "stat": "Average",
                "region": region,
                "title": "Infrastructure: CPU Utilization"
            }
        },
        {
            "type": "metric",
            "x": 12, "y": 0, "width": 12, "height": 6,
            "properties": {
                "metrics": [
                    ["AWS/SageMaker", "MemoryUtilization", "EndpointName", endpoint_name, "VariantName", "AllInstances"]
                ],
                "period": 300,
                "stat": "Average",
                "region": region,
                "title": "Infrastructure: Memory Utilization"
            }
        }
    ]
}

cw_client.put_dashboard(
    DashboardName='NYC-Collision-ML-Performance',
    DashboardBody=pd.io.json.dumps(dashboard_body)
)

print("CloudWatch Dashboard 'NYC-Collision-ML-Performance' created successfully.")

## 5. Final Monitoring Report
| Requirement | Status | Metric Location |
| :--- | :--- | :--- |
| Data Monitor | ✅ Active | SageMaker Model Monitor (S3) |
| Model Monitor | ✅ Active | SageMaker Model Monitor (S3) |
| Infra Monitor | ✅ Active | CloudWatch Metrics |
| Dashboard | ✅ Created | CloudWatch Dashboards |
| Reports | ✅ Generated | SageMaker Console / S3 |

**Project Tracker Updated:** [Done] Implementation of Monitoring Suite.

In [ ]:
# Cleanup: Uncomment to delete endpoint and avoid costs
# predictor.delete_endpoint()
# predictor.delete_model()